<a href="https://colab.research.google.com/github/nirajchavhan31-dotcom/bigdata/blob/main/AI_Travel_Planner_Final_Year.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip -q install google-genai requests folium reportlab ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 93.1 MB/s eta 0:00:00


In [5]:

import os
import re
import json
import hmac
import hashlib
import secrets
import sqlite3
import requests
import folium
import ipywidgets as widgets

from datetime import datetime
from IPython.display import display, clear_output
from google.colab import files
from google import genai
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
)
from xml.sax.saxutils import escape

# -------------------- DATABASE --------------------

def db_connect():
    return sqlite3.connect(DB_PATH, timeout=30)

def initialize_database():
    with db_connect() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT UNIQUE NOT NULL,
                salt TEXT NOT NULL,
                password_hash TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS trips (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                user_id INTEGER NOT NULL,
                destination TEXT NOT NULL,
                days INTEGER NOT NULL,
                budget REAL NOT NULL,
                interests TEXT,
                itinerary TEXT NOT NULL,
                created_at TEXT NOT NULL,
                FOREIGN KEY(user_id) REFERENCES users(id)
            )
        """)

initialize_database()

def hash_password(password, salt):
    return hashlib.pbkdf2_hmac(
        "sha256", password.encode("utf-8"),
        bytes.fromhex(salt), 200000
    ).hex()

def register_user(username, password):
    username = username.strip().lower()

    if not re.fullmatch(r"[a-z0-9_]{3,30}", username):
        return False, "Username: 3-30 letters, numbers or underscores."

    if len(password) < 8:
        return False, "Password must contain at least 8 characters."

    salt = secrets.token_hex(16)
    password_hash = hash_password(password, salt)

    try:
        with db_connect() as conn:
            conn.execute(
                """INSERT INTO users
                   (username, salt, password_hash, created_at)
                   VALUES (?, ?, ?, ?)""",
                (username, salt, password_hash,
                 datetime.now().isoformat(timespec="seconds"))
            )
        return True, "Account created. You can now log in."
    except sqlite3.IntegrityError:
        return False, "Username already exists."

def authenticate_user(username, password):
    with db_connect() as conn:
        row = conn.execute(
            """SELECT id, salt, password_hash FROM users
               WHERE username = ?""",
            (username.strip().lower(),)
        ).fetchone()

    if not row:
        return None

    user_id, salt, stored_hash = row
    candidate_hash = hash_password(password, salt)

    if hmac.compare_digest(candidate_hash, stored_hash):
        return user_id
    return None

def save_trip(user_id, destination, days, budget, interests, itinerary):
    with db_connect() as conn:
        conn.execute(
            """INSERT INTO trips
               (user_id, destination, days, budget, interests,
                itinerary, created_at)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (user_id, destination, days, budget,
             json.dumps(interests),
             itinerary,
             datetime.now().isoformat(timespec="seconds"))
        )

def get_trip_history(user_id):
    with db_connect() as conn:
        return conn.execute(
            """SELECT id, destination, days, budget, created_at
               FROM trips WHERE user_id = ?
               ORDER BY id DESC LIMIT 20""",
            (user_id,)
        ).fetchall()

# -------------------- WEATHER AND GEOCODING --------------------

def geocode(place):
    response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": place, "count": 1, "language": "en"},
        timeout=20
    )
    response.raise_for_status()
    results = response.json().get("results", [])

    if not results:
        raise ValueError("Location not found. Check the city name.")

    item = results[0]
    return {
        "name": item["name"],
        "country": item.get("country", ""),
        "lat": item["latitude"],
        "lon": item["longitude"]
    }

def get_weather(location):
    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": location["lat"],
            "longitude": location["lon"],
            "current": "temperature_2m,relative_humidity_2m,"
                       "apparent_temperature,is_day,weather_code,"
                       "wind_speed_10m",
            "daily": "temperature_2m_max,temperature_2m_min,"
                     "precipitation_probability_max",
            "forecast_days": 3,
            "timezone": "auto"
        },
        timeout=20
    )
    response.raise_for_status()
    return response.json()

def weather_summary(data):
    current = data.get("current", {})
    daily = data.get("daily", {})

    lines = [
        "Current temperature: {} C".format(
            current.get("temperature_2m", "N/A")
        ),
        "Feels like: {} C".format(
            current.get("apparent_temperature", "N/A")
        ),
        "Humidity: {}%".format(
            current.get("relative_humidity_2m", "N/A")
        ),
        "Wind speed: {} km/h".format(
            current.get("wind_speed_10m", "N/A")
        ),
        "",
        "Available forecast:"
    ]

    dates = daily.get("time", [])
    highs = daily.get("temperature_2m_max", [])
    lows = daily.get("temperature_2m_min", [])
    rain = daily.get("precipitation_probability_max", [])

    for i, date in enumerate(dates):
        lines.append(
            f"{date}: low {lows[i]} C, high {highs[i]} C, "
            f"precipitation probability {rain[i]}%"
        )

    return "\n".join(lines)

# -------------------- AI TRAVEL PLANNER --------------------

def generate_ai_response(prompt):
    if not GEMINI_API_KEY:
        return None

    client = genai.Client(api_key=GEMINI_API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

def create_itinerary(destination, days, budget, travelers, interests,
                     style, transport, accommodation, weather_text):
    prompt = f"""
You are an AI travel planning assistant.
Create a practical itinerary in simple English.

Destination: {destination}
Trip duration: {days} days
Total trip budget in INR: {budget}
Number of travelers: {travelers}
Interests: {", ".join(interests) or "General sightseeing"}
Travel style: {style}
Preferred transport: {transport}
Accommodation: {accommodation}

Weather information retrieved for this destination:
{weather_text}

Include:
1. A short destination introduction.
2. Each day with morning, afternoon and evening activities.
3. Places to visit and local food suggestions.
4. Estimated budget split in INR.
5. Transport and accommodation tips.
6. Safety advice and weather-related suggestions.
7. Explain if the budget is unlikely to be sufficient.

Keep the estimated total within budget if reasonably possible.
Do not claim prices, attraction hours or bookings are verified.
Clearly identify estimates. Use headings and readable bullet points.
"""
    answer = generate_ai_response(prompt)

    if answer:
        return answer

    # Offline/API-key-free demonstration fallback
    output = [
        f"TRAVEL PLAN: {destination}",
        f"Duration: {days} days",
        f"Budget: INR {budget:,.0f}",
        f"Travelers: {travelers}",
        f"Travel style: {style}",
        "",
        "DEMO MODE: Add a valid Gemini API key for AI-generated plans.",
        ""
    ]

    for day in range(1, days + 1):
        output.extend([
            f"DAY {day}",
            "Morning: Visit a major local attraction.",
            "Afternoon: Explore nearby sights and enjoy local food.",
            "Evening: Explore a market or a scenic location.",
            "Check local opening hours, travel times and ticket prices.",
            ""
        ])

    output.extend([
        "INTERESTS: " + ", ".join(interests),
        "Transport: " + transport,
        "Accommodation: " + accommodation,
        "",
        "Estimated allocation (illustrative, not a live quote):",
        f"Transport: INR {budget * 0.25:,.0f}",
        f"Accommodation: INR {budget * 0.30:,.0f}",
        f"Food: INR {budget * 0.20:,.0f}",
        f"Activities: INR {budget * 0.15:,.0f}",
        f"Other expenses: INR {budget * 0.10:,.0f}",
        "",
        "Confirm weather, prices, transport and safety information "
        "before travel."
    ])
    return "\n".join(output)

# -------------------- MAP --------------------

def create_trip_map(destination, attraction_text):
    main_location = geocode(destination)

    travel_map = folium.Map(
        location=[main_location["lat"], main_location["lon"]],
        zoom_start=11
    )

    folium.Marker(
        [main_location["lat"], main_location["lon"]],
        popup=main_location["name"] + ", " + main_location["country"],
        tooltip="Main destination",
        icon=folium.Icon(color="blue", icon="info-sign")
    ).add_to(travel_map)

    locations = [[main_location["lat"], main_location["lon"]]]

    attractions = [
        item.strip() for item in attraction_text.split(",")
        if item.strip()
    ]

    for attraction in attractions[:8]:
        try:
            place = geocode(attraction + ", " + destination)
            coords = [place["lat"], place["lon"]]

            folium.Marker(
                coords,
                popup=place["name"],
                tooltip=place["name"],
                icon=folium.Icon(color="green", icon="map-marker")
            ).add_to(travel_map)

            locations.append(coords)
        except Exception:
            print("Could not locate attraction:", attraction)

    if len(locations) > 1:
        folium.PolyLine(
            locations, color="blue", weight=3, opacity=0.7
        ).add_to(travel_map)

    return travel_map, main_location

# -------------------- PDF REPORT --------------------

def export_pdf(destination, days, budget, travelers,
               interests, itinerary, weather_text):
    safe_name = re.sub(
        r"[^a-zA-Z0-9_-]+", "_", destination
    ).strip("_") or "trip"

    output_path = str(
        PROJECT_DIR / f"travel_report_{safe_name}.pdf"
    )

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=40
    )

    styles = getSampleStyleSheet()
    story = [
        Paragraph("AI TRAVEL PLANNER", styles["Title"]),
        Paragraph("Personalized Travel Report", styles["Heading2"]),
        Spacer(1, 12)
    ]

    details = [
        ["Destination", destination],
        ["Duration", f"{days} days"],
        ["Budget", f"INR {budget:,.0f}"],
        ["Travelers", str(travelers)],
        ["Interests", ", ".join(interests) or "General"]
    ]

    table = Table(details, colWidths=[110, 370])
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, -1), colors.lightblue),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("PADDING", (0, 0), (-1, -1), 7)
    ]))
    story.extend([table, Spacer(1, 16)])

    story.append(Paragraph("Travel Itinerary", styles["Heading1"]))

    for line in itinerary.splitlines():
        line = line.strip()

        if not line:
            story.append(Spacer(1, 5))
        elif line.startswith("#"):
            heading = line.lstrip("# ").strip()
            story.append(Paragraph(escape(heading), styles["Heading2"]))
        else:
            line = re.sub(r"\*\*(.*?)\*\*", r"\1", line)
            story.append(Paragraph(escape(line), styles["BodyText"]))

    story.extend([
        Spacer(1, 16),
        Paragraph("Weather Summary", styles["Heading1"])
    ])

    for line in weather_text.splitlines():
        if line.strip():
            story.append(Paragraph(escape(line), styles["BodyText"]))
        else:
            story.append(Spacer(1, 5))

    story.extend([
        Spacer(1, 20),
        Paragraph(
            "Note: AI suggestions and estimated costs are not confirmed "
            "bookings. Verify weather, local advisories, entry fees and "
            "transport before travelling.",
            styles["Italic"]
        )
    ])

    doc.build(story)
    return output_path

# -------------------- USER INTERFACE --------------------

user_state = {"id": None, "username": None}
trip_state = {"plan": "", "weather": "", "destination": "",
              "days": 3, "budget": 10000, "travelers": 1,
              "interests": []}

auth_output = widgets.Output()
app_output = widgets.Output()

username_input = widgets.Text(
    description="Username:", placeholder="Create a username"
)
password_input = widgets.Password(
    description="Password:", placeholder="At least 8 characters"
)
register_button = widgets.Button(
    description="Register", button_style="info"
)
login_button = widgets.Button(
    description="Login", button_style="success"
)

def on_register(_):
    with auth_output:
        clear_output()
        ok, message = register_user(
            username_input.value, password_input.value
        )
        print(message)

def on_login(_):
    with auth_output:
        clear_output()
        user_id = authenticate_user(
            username_input.value, password_input.value
        )

        if user_id is None:
            print("Login failed. Check your username and password.")
            return

        user_state["id"] = user_id
        user_state["username"] = username_input.value.strip().lower()
        print("Login successful. Welcome,", user_state["username"])

    show_planner()

register_button.on_click(on_register)
login_button.on_click(on_login)

display(widgets.HTML("<h2>✈️ AI Travel Planner Chatbot</h2>"))
display(widgets.HTML("<b>Register a new account or log in</b>"))
display(username_input, password_input)
display(widgets.HBox([register_button, login_button]), auth_output)

def show_planner():
    with app_output:
        clear_output(wait=True)

        destination_input = widgets.Text(
            description="Destination:",
            placeholder="e.g. Goa, Manali, Jaipur",
            layout=widgets.Layout(width="95%")
        )
        days_input = widgets.IntSlider(
            value=3, min=1, max=15,
            description="Days:", continuous_update=False
        )
        budget_input = widgets.IntText(
            value=10000, description="Budget ₹:"
        )
        travelers_input = widgets.IntSlider(
            value=2, min=1, max=20, description="Travelers:"
        )
        style_input = widgets.Dropdown(
            options=[
                "Budget-friendly", "Adventure", "Relaxation",
                "Family trip", "Romantic", "Cultural and historical"
            ],
            description="Style:"
        )
        transport_input = widgets.Dropdown(
            options=["Train", "Bus", "Car", "Flight", "Local transport"],
            description="Transport:"
        )
        stay_input = widgets.Dropdown(
            options=[
                "Budget hotel", "Hostel", "3-star hotel", "Luxury hotel"
            ],
            description="Stay:"
        )
        interests_input = widgets.SelectMultiple(
            options=[
                "Nature", "Beaches", "Mountains", "History", "Food",
                "Shopping", "Adventure", "Photography", "Wildlife"
            ],
            value=("Nature", "Food"),
            description="Interests:",
            layout=widgets.Layout(width="95%", height="130px")
        )
        attractions_input = widgets.Text(
            description="Places:",
            placeholder="Optional: attraction 1, attraction 2",
            layout=widgets.Layout(width="95%")
        )

        generate_button = widgets.Button(
            description="Generate AI Travel Plan",
            button_style="success",
            icon="paper-plane"
        )
        weather_button = widgets.Button(
            description="Get Weather", icon="cloud"
        )
        map_button = widgets.Button(
            description="Show Map", icon="map"
        )
        pdf_button = widgets.Button(
            description="Download PDF", icon="download"
        )
        history_button = widgets.Button(
            description="Trip History", icon="history"
        )
        logout_button = widgets.Button(
            description="Logout", button_style="warning"
        )

        plan_output = widgets.Output()
        weather_output = widgets.Output()
        map_output = widgets.Output()
        chat_output = widgets.Output()
        history_output = widgets.Output()

        followup_input = widgets.Text(
            description="Ask:",
            placeholder="e.g. Make Day 2 cheaper",
            layout=widgets.Layout(width="85%")
        )
        followup_button = widgets.Button(
            description="Send",
            button_style="info"
        )

        def read_trip_inputs():
            destination = destination_input.value.strip()
            days = days_input.value
            budget = budget_input.value
            travelers = travelers_input.value
            interests = list(interests_input.value)

            if not destination:
                raise ValueError("Enter a destination.")
            if budget < 1000:
                raise ValueError("Budget must be at least INR 1000.")

            return destination, days, budget, travelers, interests

        def get_weather_for_trip(destination):
            location = geocode(destination)
            data = get_weather(location)
            return location, data, weather_summary(data)

        def on_weather(_):
            with weather_output:
                clear_output()
                try:
                    destination, *_ = read_trip_inputs()
                    location, data, summary = get_weather_for_trip(destination)
                    trip_state["weather"] = summary
                    print(
                        f"Weather for {location['name']}, "
                        f"{location['country']}"
                    )
                    print(summary)
                    print("\nWeather source: Open-Meteo")
                except Exception as exc:
                    print("Weather error:", exc)

        def on_map(_):
            with map_output:
                clear_output()
                try:
                    destination, *_ = read_trip_inputs()
                    travel_map, location = create_trip_map(
                        destination, attractions_input.value
                    )
                    print(
                        "Map for", location["name"],
                        "- attraction markers are shown when locations resolve."
                    )
                    display(travel_map)
                except Exception as exc:
                    print("Map error:", exc)

        def on_generate(_):
            with plan_output:
                clear_output()
                try:
                    (destination, days, budget,
                     travelers, interests) = read_trip_inputs()

                    weather_text = "Weather not fetched yet."
                    try:
                        location, data, weather_text = (
                            get_weather_for_trip(destination)
                        )
                        trip_state["weather"] = weather_text
                    except Exception as exc:
                        weather_text = f"Weather unavailable: {exc}"

                    plan = create_itinerary(
                        destination, days, budget, travelers,
                        interests, style_input.value,
                        transport_input.value, stay_input.value,
                        weather_text
                    )

                    trip_state.update({
                        "plan": plan,
                        "destination": destination,
                        "days": days,
                        "budget": budget,
                        "travelers": travelers,
                        "interests": interests
                    })

                    save_trip(
                        user_state["id"], destination, days,
                        budget, interests, plan
                    )

                    print(plan)
                    print("\nTrip saved to your database.")

                except Exception as exc:
                    print("Could not generate itinerary:", exc)

        def on_followup(_):
            with chat_output:
                clear_output()
                question = followup_input.value.strip()

                if not trip_state["plan"]:
                    print("Generate a trip plan first.")
                    return
                if not question:
                    print("Type your question first.")
                    return

                prompt = f"""
You are the user's travel-planning chatbot.
Use the existing itinerary as context.

Existing itinerary:
{trip_state['plan']}

User follow-up:
{question}

Respond helpfully. Do not claim bookings or live prices are confirmed.
"""
                try:
                    answer = generate_ai_response(prompt)
                    if not answer:
                        answer = (
                            "Demo mode: add a Gemini API key for intelligent "
                            "follow-up responses. Your question was: " + question
                        )
                    print("You:", question)
                    print("\nTravel Assistant:", answer)
                except Exception as exc:
                    print("Chatbot error:", exc)

        def on_pdf(_):
            with plan_output:
                if not trip_state["plan"]:
                    print("Generate a trip plan before downloading the PDF.")
                    return
                try:
                    path = export_pdf(
                        trip_state["destination"],
                        trip_state["days"],
                        trip_state["budget"],
                        trip_state["travelers"],
                        trip_state["interests"],
                        trip_state["plan"],
                        trip_state["weather"] or "Weather not available."
                    )
                    print("PDF created:", path)
                    files.download(path)
                except Exception as exc:
                    print("PDF error:", exc)

        def on_history(_):
            with history_output:
                clear_output()
                rows = get_trip_history(user_state["id"])
                if not rows:
                    print("No saved trips yet.")
                else:
                    print("Recent saved trips:")
                    for row in rows:
                        print(
                            f"ID {row[0]} | {row[1]} | {row[2]} days | "
                            f"₹{row[3]:,.0f} | {row[4]}"
                        )

        def on_logout(_):
            user_state["id"] = None
            user_state["username"] = None
            trip_state["plan"] = ""
            with app_output:
                clear_output()
                print("Logged out. Run the login button to sign in again.")

        generate_button.on_click(on_generate)
        weather_button.on_click(on_weather)
        map_button.on_click(on_map)
        pdf_button.on_click(on_pdf)
        followup_button.on_click(on_followup)
        history_button.on_click(on_history)
        logout_button.on_click(on_logout)

        display(widgets.HTML(
            f"<h3>Welcome, {escape(user_state['username'])}!</h3>"
        ))
        display(destination_input, days_input, budget_input)
        display(travelers_input, style_input)
        display(interests_input, transport_input, stay_input)
        display(attractions_input)
        display(widgets.HBox([
            generate_button, weather_button, map_button
        ]))
        display(plan_output, weather_output, map_output)
        display(widgets.HTML("<h3>💬 Ask a follow-up question</h3>"))
        display(widgets.HBox([followup_input, followup_button]))
        display(chat_output)
        display(widgets.HBox([pdf_button, history_button, logout_button]))
        display(history_output)

display(app_output)


HTML(value='<h2>✈️ AI Travel Planner Chatbot</h2>')

HTML(value='<b>Register a new account or log in</b>')

Text(value='', description='Username:', placeholder='Create a username')

Password(description='Password:', placeholder='At least 8 characters')

Output()

Output()